<center><h1>Feature Engineering</h1>
After looking at the quality of the data through exploratory data analysis, we will apply data cleaning and processing.</center>

## Import package and Load Processed Data

In [37]:
import pandas as pd

In [38]:
df = pd.read_csv("df_processed.csv")

## Rolling Metric
Since all numerical features, like `dpm` or `kills`, are data we will only know after the game had concluded, there will be data leakage if we use same-day game stats to predict the same game result. To circumvent data leakage, we will implement a rolling 10-step average for each metric, with a minimum of 1-step window for teams that has smaller historical data.

In [39]:
# Sort chronologically
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['date']).reset_index(drop=True)

In [40]:
# Calculate rolling 10-match averages grouped by team
rolling_cols = ["kills", "deaths", "dpm", "damagetakenperminute", "totalgold"]

for col in rolling_cols:
  col_name = f"{col}_rolling_avg_10"
  # Group by team and calculate rolling mean, shifting by 1 so we don't leak the current match stats
  df[col_name] = (
      df.groupby("teamname")[col]
      .shift(1)
      .rolling(window=10, min_periods=1)
      .mean()
  )

In [42]:
# Drop rows where rolling features are NaN (i.e., each team's first match)
df = df.dropna().reset_index(drop=True)

## Data Readiness
The data is almost ready for modeling. We will just have to remove the identifiers like `gameid` and `date`, and drop same-day features like `kills` and `deaths`. We only need the rolling averages for modeling. Finally, we will encode `split` for chronological validation splitting during modeling.

In [43]:
# Drop identifiers
df = df.drop(['gameid', 'league', 'date', 'teamname'], axis='columns', errors='ignore')

In [44]:
# Dropping same-day features
df = df.drop(['kills', 'deaths', 'dpm', 'damagetakenperminute', 'wardskilled', 'totalgold'], axis='columns', errors='ignore')

In [45]:
# One-Hot Enconde split and dropping the first column to remove multicollinearity
df = pd.get_dummies(df, columns=['split'], drop_first=True)

In [46]:
# Quick snapshot of the dataframe ready for modeling
df.head()

,result,kills_rolling_avg_10,deaths_rolling_avg_10,dpm_rolling_avg_10,damagetakenperminute_rolling_avg_10,totalgold_rolling_avg_10,split_Summer
0,0,9.000000,14.000000,2007.623300,2986.734000,54724.000000,False
1,1,15.000000,9.500000,2155.287750,2741.594650,57077.500000,False
2,1,11.666667,13.333333,2002.056000,2952.386733,53126.666667,False
3,0,12.250000,12.250000,2089.458325,2909.473975,57557.750000,False
4,0,12.250000,12.250000,2089.458325,2909.473975,57557.750000,False


In [47]:
# df.to_csv("df_ready.csv", index=False)